# Employee Job Review Analysis

Notebook reorganized in the same block order, with the main model changed from binary classification to a true 3-class setup: positive, negative, and mixed.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report


: 

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")


In [ ]:
df = pd.read_csv("glassdoor_reviews.csv")

df.columns = df.columns.str.lower().str.replace("-", "_")

df = df[["firm", "pros", "cons", "overall_rating"]].dropna()
df.columns = ["company", "pros", "cons", "rating"]

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df[df["rating"].between(1, 5)]

df.head()


In [ ]:
rating_counts = df["rating"].value_counts().sort_index()

rating_counts.plot(kind="bar")

plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

extra = {"company","work","job","people","get","make","also","really"}
stop_words.update(extra)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 2
    ]
    return " ".join(tokens)

df["combined"] = (df["pros"] + " " + df["cons"]).apply(clean_text)

df[["combined"]].head()


In [ ]:
analyzer = SentimentIntensityAnalyzer()

df["sentiment_score"] = df["combined"].apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)

df[["sentiment_score"]].describe()


In [ ]:
# =============================================================================
# GRAPH 2: SENTIMENT DISTRIBUTION
# =============================================================================

df["sentiment_score"].plot(
    kind="hist",
    bins=50,
    edgecolor="black"
)

plt.title("Sentiment Score Distribution")
plt.xlabel("Sentiment Score (-1 to +1)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=5)
X_tfidf = tfidf.fit_transform(df["combined"])


In [ ]:
print("\n--- Preparing LDA ---")

vectorizer_lda = CountVectorizer(max_features=5000, stop_words="english")
X_lda = vectorizer_lda.fit_transform(df["combined"])

print(f"Vocabulary size: {len(vectorizer_lda.get_feature_names_out())}")

lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="batch",
    max_iter=5
)

lda_model.fit(X_lda)

def print_topics(model, feature_names, n_words=8):
    print("\n--- LDA Topics ---")
    for idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[-n_words:][::-1]]
        print(f"\nTopic {idx + 1}: " + " + ".join(top_words))

print_topics(lda_model, vectorizer_lda.get_feature_names_out())

df["topic"] = lda_model.transform(X_lda).argmax(axis=1)


In [ ]:
# =============================================================================
# GRAPH 3: TOPIC DISTRIBUTION
# =============================================================================

df["topic"].value_counts().sort_index().plot(
    kind="bar",
    edgecolor="black"
)

plt.title("Distribution of Topics in Reviews")
plt.xlabel("Topic")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Main model: 3-class classification

Rating 4-5 → positive, rating 1-2 → negative, rating 3 → mixed.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay

df_clf = df.copy()

def label_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating <= 2:
        return "negative"
    else:
        return "mixed"

df_clf["label"] = df_clf["rating"].apply(label_sentiment)

le = LabelEncoder()
y = le.fit_transform(df_clf["label"])

X = tfidf.transform(df_clf["combined"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
# =============================================================================
# LOGISTIC REGRESSION MODEL (3-CLASS VERSION)
# =============================================================================

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs"
)

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print("LOGISTIC REGRESSION RESULTS")
print(classification_report(y_test, pred_lr, target_names=le.classes_))


In [ ]:
# =============================================================================
# CONFUSION MATRIX - LOGISTIC REGRESSION
# =============================================================================

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_lr,
    display_labels=le.classes_
)

plt.title("Confusion Matrix - Logistic Regression")
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

lr_cv_scores = cross_val_score(
    lr,
    X,
    y,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1
)

print("LOGISTIC REGRESSION CROSS-VALIDATION")
print(f"F1 Score: {lr_cv_scores.mean():.3f} ± {lr_cv_scores.std():.3f}")


In [ ]:
# =============================================================================
# JUDGE-STYLE SENTIMENT ANALYZER (ML + VADER FUSION)
# =============================================================================

def predict_review(review_text):

    text = clean_text(review_text)

    # VADER
    vader_score = analyzer.polarity_scores(review_text)["compound"]

    if vader_score >= 0.05:
        vader_label = "Positive 😊"
    elif vader_score <= -0.05:
        vader_label = "Negative 😞"
    else:
        vader_label = "Mixed / Neutral 😐"

    # ML prediction
    vec = tfidf.transform([text])
    ml_pred = lr.predict(vec)[0]
    ml_proba = lr.predict_proba(vec)[0]

    ml_label = le.inverse_transform([ml_pred])[0].capitalize()

    class_index = {label: idx for idx, label in enumerate(le.classes_)}

    pos_score = ml_proba[class_index["positive"]] if "positive" in class_index else 0
    neg_score = ml_proba[class_index["negative"]] if "negative" in class_index else 0

    ml_score = pos_score - neg_score

    # Fusion
    final_score = (0.6 * vader_score) + (0.4 * ml_score)

    if final_score > 0.1:
        final_label = "FINAL: Positive ✅"
    elif final_score < -0.1:
        final_label = "FINAL: Negative ❌"
    else:
        final_label = "FINAL: Mixed ⚖️"

    # Output
    print("\n" + "═" * 60)
    print("🧠 SENTIMENT JUDGE SYSTEM")
    print("═" * 60)

    print(f"Input Review: {review_text}")

    print("\n--- VADER ---")
    print(f"Score: {vader_score:+.3f} → {vader_label}")

    print("\n--- ML Model ---")
    print(f"Prediction: {ml_label}")

    print("\n--- FINAL DECISION ---")
    print(final_label)

    print("═" * 60)


In [ ]:
predict_review("poor management but flexible hours and good colleagues")
